In [ ]:
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from openai import OpenAI

# Constants
SPREADSHEET_ID = '1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc'  
SHEET_NAME = 'Sheet2'
CREDENTIALS_FILE = 'url-to-email-445616-cebe4868914f.json'  
OPENAI_API_KEY = ''  
# Category ranking
category_ranking = {
    "TESTIMONIALS": 1,
    "COURSES": 2,
    "SERVICES": 3,
    "WEBINAR": 4,
    "PODCAST": 5,
    "EBOOK": 6,
    "RECENT_BLOG": 7,
    "ABOUT_US": 8
}

# Column indices for categories (0-based, A=0, B=1, ..., M=12, N=13, ..., U=20)
category_indices = {
    "ABOUT_US": 12,  # M
    "EBOOK": 13,     # N
    "COURSES": 14,   # O
    "RECENT_BLOG": 15, # P
    "TESTIMONIALS": 16, # Q
    "WEBINAR": 17,   # R
    "SERVICES": 18,  # S
    "PODCAST": 19,   # T
    "SHOP": 20       # U (excluded)
}

# Output column letters
output_columns = {
    "Email 1": "W",
    "Email 1 Data Point": "X",
    "Subsequence 1": "Y",
    "Subsequence 1 Data Point": "Z",
    "Subsequence 2": "AA",
    "Subsequence 2 Data Point": "AB",
    "Subsequence 3": "AC",
    "Subsequence 3 Data Point": "AD",
    "Subsequence 4": "AE",
    "Subsequence 4 Data Point": "AF",
    "Email 2": "AG",
    "Email 3": "AH"
}

# Industry problem mapping
industry_problems = {
    "SaaS": "scaling growth",
    "Fintech": "building trust",
    "E-commerce": "increasing sales",
    "default": "growing your business"
}

# Default values for placeholders
default_values = {
    "First Name": "Hi there",
    "Industry": "SaaS",
    "Company Name": "your company",
    "Website": "your website"
}

# Prompt templates
prompt_templates = {
    "Email 1": {
        "TESTIMONIALS": """[First Name], your story about [Personalization] impressed me—seeing how you helped your clients achieve such tangible results is inspiring.
I believe showcasing this case study to our community of [Industry] leaders could spark the same transformation for countless others.
If we partnered to feature your success prominently on our specialized partner publications, I’m confident we could drive a surge in qualified interest within 45 days—no extra work on your end.
Curious to explore how this could amplify your client impact?

Warmly,
[Your Name]""",
        "COURSES": """[First Name], I just finished your [Personalization] course—its practical frameworks are exactly what professionals in [Industry] are craving.
Imagine those insights reaching our network of active learners and decision-makers through a tailored spotlight article.
In the next month and a half, you could see a significant uptick in enrollments, without lifting a finger.
Shall we chat about making this happen?

Best,
[Your Name]""",
        "SERVICES": """[First Name], I explored your [Personalization] service page—your approach to [Industry Problem] is exactly what the market needs.
What if we crafted a featured story around your methodology and shared it with our community of [Industry] professionals?
In 45 days, you could see a wave of new inquiries, and you won’t have to do any heavy lifting.
Interested in discussing the details?

Cheers,
[Your Name]""",
        "WEBINAR": """[First Name], your webinar on [Personalization] was a goldmine—your deep dive into [Specific Topic] really resonated with me.
We’d love to feature your session highlights in our monthly [Industry] Insights Digest, reaching over 10,000 decision-makers.
Picture doubling your webinar attendance within 45 days, effortlessly.
Can we set up a quick call to explore?

Thank you,
[Your Name]""",
        "PODCAST": """[First Name], I enjoyed your episode on [Personalization]—your insights on [Key Point] were spot-on.
Our audience of [Industry] executives would benefit greatly from your expertise.
If we worked together to amplify this episode through our curated partner channels, I believe we could drive significant engagement in just 45 days.
Would you like to hear how?

Regards,
[Your Name]""",
        "EBOOK": """[First Name], your ebook on [Personalization] offers a masterclass in [Topic]; I found the section on [Specific Section] particularly compelling.
Imagine reaching an even broader audience of [Industry] professionals eager for that guidance.
By featuring your ebook in our resource showcase, we could boost downloads dramatically—no extra effort on your part.
Curious to learn more?

All the best,
[Your Name]""",
        "RECENT_BLOG": """[First Name], your recent post about [Personalization] hit the nail on the head—your analysis of [Specific Topic] was enlightening.
We’d be thrilled to spotlight it in our weekly newsletter to over 5,000 readers in [Industry].
This could drive a notable uptick in readership without you lifting a finger.
Want to discuss the opportunity?

Sincerely,
[Your Name]""",
        "ABOUT_US": """[First Name], I reviewed [Company Name] and was impressed by your creative take on [Personalization].
Your approach stands out, and I know our network of [Industry] leaders would love to learn from it.
If we crafted a feature story about your journey, you could see a surge in high-quality leads within 45 days—effortlessly.
Shall we set up a brief call?

Thanks,
[Your Name]""",
        "NEUTRAL": """[First Name], I admire the work you’ve been doing with [Personalization].
Your expertise deserves greater visibility—imagine your insights reaching a targeted audience of engaged professionals.
If we collaborated to share your best material through our industry spotlight series, you could see meaningful growth in just 45 days, with zero extra effort.
Would you like to explore this?

Best regards,
[Your Name]"""
    },
    "Subsequence 1": {
        "TESTIMONIALS": """Thanks for reaching out! The work you did with [Personalization] really caught my eye.
Can you help more folks get those results? I bet they’d love it!
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "COURSES": """Thanks for reaching out! Your [Personalization] course made me think…
Can we share its ideas? People would love them!
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "SERVICES": """Thanks for reaching out! Your [Personalization] service page got my attention…
Can we share its ideas? I think people would love them!
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "WEBINAR": """Thanks for reaching out! Your [Personalization] webinar was so cool…
Can we share its ideas? People would be excited!
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "PODCAST": """Thanks for reaching out! Your podcast about [Personalization] was awesome…
Can we share its ideas? People would love them!
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "EBOOK": """Thanks for reaching out! Your book on [Personalization] was really neat…
Can we share its ideas? I think people would love them!
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "RECENT_BLOG": """Thanks for reaching out! Your blog on [Personalization] got my attention…
Can we share its ideas? People would be excited!
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "ABOUT_US": """Thanks for reaching out! I loved [Company Name]’s approach to [Personalization]…
It’s so cool! Can we share it with others?
Here’s a video I recorded going into detail on this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]""",
        "NEUTRAL": """Appreciate you reaching back out! I think there’s sooo much potential if we do this together.
Here’s a quick 2 min pre-recorded video (to save myself some time LOL) we recorded going into this: 
Loom.com/video
I’d love to find out more about [Company Name] and how this could 5-10x your exposure. 
Grab a time with me here: 
https://calendly.com/scale-brands-lab/30min
Thx, 
[Your Name]"""
    },
    "Subsequence 2": """I was looking at your [Website] and couldn’t help but notice [Personalization]. 
Got some time to talk about it tomorrow?""",
    "Subsequence 3": """Knowing that AI’s blowing up in [Industry], [First Name].
Like those crazy chatbots!
I'm curious, what’s your next move to stay ahead of the curve?
Chat soon,
[Your Name]""",
    "Subsequence 4": """I keep wondering about how [Company Name] is pushing [Personalization].
We’ve been working with some folks on similar goals, and I’d love to bounce a couple ideas your way—might be a fit. 
Got a minute to talk this week?""",
    "Email 2": "Hate to bug you. Is this something I can pass along or no?",
    "Email 3": """Hey [First Name],
Just wanted to let you know…
Being an Invite only firm, part of our offer is:
If we can’t 5x your current exposure in the next 45 days, then we work for FREE until we do.
Would it make sense to talk about it?"""
}

# Initialize Google Sheets API
creds = service_account.Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=['https://www.googleapis.com/auth/spreadsheets'])
service = build('sheets', 'v4', credentials=creds)

# Initialize OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY)

def summarize_content(content):
    """Summarize content to less than 15 words using OpenAI, or truncate if fails."""
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Summarize the following content to less than 15 words at a 3rd-grade reading level."},
                {"role": "user", "content": content}
            ],
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Summarization failed: {e}")
        words = content.split()
        return ' '.join(words[:14]) + '…' if len(words) > 14 else content

def replace_placeholders(template, data):
    """Replace placeholders in the template with data values."""
    for key, value in data.items():
        template = template.replace(f"[{key}]", value)
    return template

def generate_email(template, data):
    """Generate an email by replacing placeholders in the template."""
    return replace_placeholders(template, data)

def process_row(row):
    """Process a single row and generate emails."""
    # Extract data with defaults
    first_name = row[0] if len(row) > 0 else default_values["First Name"]
    company_name = row[3] if len(row) > 3 else default_values["Company Name"]
    industry = row[8] if len(row) > 8 else default_values["Industry"]
    website = row[6] if len(row) > 6 else default_values["Website"]

    # Collect available data points
    available_data_points = []
    for category in category_ranking:
        col_index = category_indices[category]
        content = row[col_index] if len(row) > col_index else ""
        if content and content not in ["no content", "No URL found", "No meaningful content found"]:
            available_data_points.append(category)

    # Sort by ranking
    available_data_points.sort(key=lambda x: category_ranking[x])

    # Assign to emails (up to 5)
    email_assignments = [None] * 5
    for i in range(min(5, len(available_data_points))):
        email_assignments[i] = available_data_points[i]

    # Generate emails
    emails = {}
    for i, email_type in enumerate(["Email 1", "Subsequence 1", "Subsequence 2", "Subsequence 3", "Subsequence 4"]):
        category = email_assignments[i]
        if category:
            content = row[category_indices[category]].strip()
            personalization = content if len(content.split()) < 15 else summarize_content(content)
            template = prompt_templates[email_type][category] if email_type in ["Email 1", "Subsequence 1"] else prompt_templates[email_type]
        else:
            template = prompt_templates[email_type]["NEUTRAL"] if email_type in ["Email 1", "Subsequence 1"] else prompt_templates[email_type]
            personalization = {
                "Email 1": "your professional expertise",
                "Subsequence 1": "your professional expertise",
                "Subsequence 2": "your great work",
                "Subsequence 3": "",
                "Subsequence 4": "customer engagement"
            }[email_type]

        data = {
            "First Name": first_name,
            "Company Name": company_name,
            "Industry": industry,
            "Personalization": personalization,
            "Website": website,
            "Your Name": "Scale Brands Lab"
        }
        if category == "SERVICES" and email_type in ["Email 1", "Subsequence 1"]:
            data["Industry Problem"] = industry_problems.get(industry, industry_problems["default"])
        if category in ["WEBINAR", "RECENT_BLOG"]:
            data["Specific Topic"] = personalization
        elif category == "PODCAST":
            data["Key Point"] = personalization
        elif category == "EBOOK":
            data["Topic"] = personalization
            data["Specific Section"] = personalization

        emails[email_type] = generate_email(template, data)
        emails[f"{email_type} Data Point"] = category if category else "Neutral"

    # Standard follow-ups
    emails["Email 2"] = prompt_templates["Email 2"]
    emails["Email 3"] = generate_email(prompt_templates["Email 3"], {"First Name": first_name})

    return emails

def main():
    """Main function to process rows and update Google Sheet."""
    # Read data from Sheet2 (A to U)
    range_name = f"{SHEET_NAME}!A2:U"
    result = service.spreadsheets().values().get(spreadsheetId=SPREADSHEET_ID, range=range_name).execute()
    rows = result.get('values', [])

    for i, row in enumerate(rows):
        # Pad row if necessary
        row += [''] * (21 - len(row)) if len(row) < 21 else []
        emails = process_row(row)

        # Prepare values to write
        values = [
            emails["Email 1"],
            emails["Email 1 Data Point"],
            emails["Subsequence 1"],
            emails["Subsequence 1 Data Point"],
            emails["Subsequence 2"],
            emails["Subsequence 2 Data Point"],
            emails["Subsequence 3"],
            emails["Subsequence 3 Data Point"],
            emails["Subsequence 4"],
            emails["Subsequence 4 Data Point"],
            emails["Email 2"],
            emails["Email 3"]
        ]

        # Write to Sheet2 (W to AH)
        write_range = f"{SHEET_NAME}!W{i+2}:AH{i+2}"
        body = {'values': [values]}
        service.spreadsheets().values().update(spreadsheetId=SPREADSHEET_ID, range=write_range, valueInputOption='RAW', body=body).execute()

        print("Columns updated")

if __name__ == "__main__":
    main()

Columns updated
Columns updated
Columns updated
Columns updated
Columns updated
Columns updated
Columns updated
Columns updated
Columns updated
